In [ ]:
pip install -qU docx2txt openpyxl unstructured pinecone langchain-ollama langchain-community

In [ ]:
pip install -qU langchain-pinecone

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from langchain_ollama import OllamaEmbeddings

# 4. Embed and store to Pinecone

# Initialize local Ollama embeddings
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")


In [ ]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# Initialize client
pc = Pinecone(api_key="")

index_name = "multi-doc-agent-index"

# Pass the string index_name directly to LangChain 
vector_store = PineconeVectorStore(
    index_name=index_name, 
    embedding=embeddings
)


In [ ]:

# 2. Map file extensions to their specific loaders
# Put all user-uploaded files into a folder named './user_uploads/'
loaders = {
    ".pdf": PyPDFLoader,
    ".docx": Docx2txtLoader,
    ".txt": TextLoader
}

documents = []
upload_dir = "./user_up"

# Loop through files and load them dynamically based on extension
for file in os.listdir(upload_dir):
    file_path = os.path.join(upload_dir, file)
    ext = os.path.splitext(file)[1].lower()
    
    if ext in loaders:
        print(f"Loading {file}...")
        loader = loaders[ext](file_path)
        documents.extend(loader.load())
    else:
        print(f"Skipping unsupported file type: {file}")

# 3. Chunk text universally
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)


In [ ]:
import pypdf
from langchain_core.documents import Document


# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "../example_data/nke-10k-2023.pdf"
docs = load_pdf_pages(file_path)
print(len(docs))

In [ ]:
import os
strs="RecursiveCharacterTextSplitter.ect"
print(os.path.splitext(strs)[1].lower())